In [ ]:
# CELDA 1 — Setup y carga de modelos (arquitectura v3 — sincronizada con scripts Python)
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os, joblib, warnings, unicodedata
import matplotlib.pyplot as plt
from datetime import datetime
warnings.filterwarnings('ignore')

PROJECT_PATH   = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH    = f'{PROJECT_PATH}/models'

print('Cargando modelos...')

# === Regresion: multi-modelo por concepto ===
modelos_regresion = joblib.load(f'{MODELS_PATH}/lgb_multi_concepto.joblib')
modelos_p10       = joblib.load(f'{MODELS_PATH}/lgb_p10_multi_concepto.joblib')
modelos_p90       = joblib.load(f'{MODELS_PATH}/lgb_p90_multi_concepto.joblib')
cqr_margenes      = joblib.load(f'{MODELS_PATH}/cqr_margenes_por_concepto.joblib')
regresion_medians = joblib.load(f'{MODELS_PATH}/regresion_medians.joblib')
feat_config       = joblib.load(f'{MODELS_PATH}/feature_config.joblib')

# === Corrección de sesgo en 2 capas (escalar + piecewise + proveedor×concepto) ===
bias_escalar        = joblib.load(f'{MODELS_PATH}/bias_correction_escalar.joblib')
bias_piecewise_corr = joblib.load(f'{MODELS_PATH}/bias_correction_piecewise.joblib')
bias_prov_concepto  = joblib.load(f'{MODELS_PATH}/bias_prov_concepto.joblib')
print(f'Bias escalar: {len(bias_escalar)} conceptos')
print(f'Bias piecewise: {sum(1 for v in bias_piecewise_corr.values() if isinstance(v, tuple))} conceptos con cuartiles')
print(f'Bias proveedor×concepto: {len(bias_prov_concepto)} pares')

# === Riesgo: Isolation Forest ===
iso_model    = joblib.load(f'{MODELS_PATH}/isolation_forest.joblib')
riesgo_enc   = joblib.load(f'{MODELS_PATH}/riesgo_encoder.joblib')
score_params = joblib.load(f'{MODELS_PATH}/riesgo_score_params.joblib')

# === Clustering: HDBSCAN + K-Means ===
hdb_model    = joblib.load(f'{MODELS_PATH}/hdbscan_model.joblib')
kmeans_model = joblib.load(f'{MODELS_PATH}/kmeans_k6.joblib')
clust_prep   = joblib.load(f'{MODELS_PATH}/clustering_preprocessor.joblib')
clust_pca    = joblib.load(f'{MODELS_PATH}/clustering_pca15.joblib')

# === Configuración de features (keys v3 — features_num / features_cat) ===
NUMERICAS   = feat_config['features_num']
CATEGORICAS = feat_config['features_cat']
FEATURES    = NUMERICAS + CATEGORICAS
COL_PROV    = feat_config.get('col_prov', 'proveedor_norm')

MEDIANA_GLOBAL_LOG    = feat_config.get('mediana_global_log', 7.0)
_TE_CI_MAP            = feat_config.get('te_ci_map', {})
_TE_CI_GLOBAL         = feat_config.get('te_ci_global', MEDIANA_GLOBAL_LOG)
_TE_CR_MAP            = feat_config.get('te_cr_map', {})
_TE_CR_GLOBAL         = feat_config.get('te_cr_global', MEDIANA_GLOBAL_LOG)
_TE_PROV_MAP          = feat_config.get('te_prov_map', {})
_TE_PROV_GLOBAL       = feat_config.get('te_prov_global', MEDIANA_GLOBAL_LOG)
_MEDIANA_POR_CONCEPTO = feat_config.get('mediana_por_concepto', {})
_CONCEPTO_CV          = feat_config.get('concepto_cv', {})

CONCEPTOS = sorted(modelos_regresion.keys())

# === Historial de train para lookups ===
hist = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
hist = hist[hist['importe_total_pen'] > 0]
if 'fecha_trabajo' in hist.columns:
    hist = hist.sort_values('fecha_trabajo')

# Precomputar niveles de categorias del training (LightGBM requiere niveles exactos)
CAT_LEVELS = {}
for col in CATEGORICAS:
    if col in hist.columns:
        vals = sorted(hist[col].fillna('DESCONOCIDO').astype(str).unique().tolist())
        if 'DESCONOCIDO' not in vals:
            vals = ['DESCONOCIDO'] + vals
        CAT_LEVELS[col] = vals
    else:
        CAT_LEVELS[col] = ['DESCONOCIDO']

print(f'Modelos de regresion: {len(modelos_regresion)} conceptos')
print(f'Features: {len(FEATURES)} ({len(NUMERICAS)} num + {len(CATEGORICAS)} cat)')
print(f'TE maps: {len(_TE_CI_MAP)} ci | {len(_TE_PROV_MAP)} proveedores')

In [ ]:
# CELDA 2 — Precomputar lookups históricos (EWM, lags 1/2/3, rolling3, TEs, N_OBS)
# Sincronizado con demo_prediccion.py v3 — todos los lookups desde datos reales del train.

def normalizar(s):
    if pd.isna(s) or s is None:
        return 'DESCONOCIDO'
    s = str(s).upper().strip()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return ' '.join(s.split())

FECHA_INICIO = pd.Timestamp('2020-01-01')

# Diccionarios de lookup
TARIFAS_PROV_CONCEPTO  = {}
TARIFAS_CONCEPTO       = {}
TARIFA_GLOBAL          = MEDIANA_GLOBAL_LOG
LAG1_PROV_CONCEPTO     = {}
LAG1_CONCEPTO          = {}
LAG2_PROV_CONCEPTO     = {}
LAG3_PROV_CONCEPTO     = {}
ROLLING3_PROV_CONCEPTO = {}
FRECUENCIA_PROVEEDOR   = {}
N_OBS_PROV_CONCEPTO    = {}

# TE maps precalculados (fit en train only) — se usan directamente, sin recalcular
TE_PROVEEDOR         = {str(k).upper(): v for k, v in _TE_PROV_MAP.items()}
TE_CONCEPTO_INCOTERM = _TE_CI_MAP
TE_CONCEPTO_RUTA     = _TE_CR_MAP
MEDIANA_POR_CONCEPTO = {k: float(v) for k, v in _MEDIANA_POR_CONCEPTO.items()}
CONCEPTO_CV          = {k: float(v) for k, v in _CONCEPTO_CV.items()}
MEDIANA_CONCEPTO_MODE = {}

col_prov = COL_PROV if COL_PROV in hist.columns else 'proveedor'
hist['log_target'] = np.log1p(hist['importe_total_pen'])

# EWM y lags por proveedor × concepto
for (prov, concepto), grupo in hist.groupby([col_prov, 'concepto_canonico']):
    prov_key = (str(prov).upper(), concepto)
    if 'fecha_trabajo' in grupo.columns:
        vals = grupo['log_target'].ewm(
            halflife='365D', times=pd.DatetimeIndex(grupo['fecha_trabajo']), min_periods=1
        ).mean()
    else:
        vals = grupo['log_target'].ewm(halflife=365, min_periods=1).mean()
    TARIFAS_PROV_CONCEPTO[prov_key] = float(vals.iloc[-1])
    hist_vals = grupo['log_target'].values
    LAG1_PROV_CONCEPTO[prov_key]     = float(hist_vals[-1])
    LAG2_PROV_CONCEPTO[prov_key]     = float(hist_vals[-2]) if len(hist_vals) >= 2 else float(hist_vals[-1])
    LAG3_PROV_CONCEPTO[prov_key]     = float(hist_vals[-3]) if len(hist_vals) >= 3 else float(hist_vals[-1])
    ROLLING3_PROV_CONCEPTO[prov_key] = float(grupo['log_target'].tail(3).median())
    N_OBS_PROV_CONCEPTO[prov_key]    = int(len(hist_vals))

# EWM y lag por concepto (fallback nivel 2)
for concepto, grupo in hist.groupby('concepto_canonico'):
    if 'fecha_trabajo' in grupo.columns:
        vals = grupo['log_target'].ewm(
            halflife='365D', times=pd.DatetimeIndex(grupo['fecha_trabajo']), min_periods=1
        ).mean()
    else:
        vals = grupo['log_target'].ewm(halflife=365, min_periods=1).mean()
    TARIFAS_CONCEPTO[concepto] = float(vals.iloc[-1])
    LAG1_CONCEPTO[concepto]    = float(grupo['log_target'].iloc[-1])

TARIFA_GLOBAL = float(hist['log_target'].ewm(halflife=365, min_periods=1).mean().iloc[-1])

FRECUENCIA_PROVEEDOR = {
    str(k).upper(): int(v)
    for k, v in hist.groupby(col_prov).size().items()
}

if 'mode' in hist.columns:
    MEDIANA_CONCEPTO_MODE = (
        hist.groupby(['concepto_canonico', 'mode'])['log_target'].median().to_dict()
    )

print(f'Tarifas prov×concepto  : {len(TARIFAS_PROV_CONCEPTO)}')
print(f'Tarifas por concepto   : {len(TARIFAS_CONCEPTO)}')
print(f'Proveedores en historial: {len(FRECUENCIA_PROVEEDOR)}')
print(f'N_OBS pares registrados: {len(N_OBS_PROV_CONCEPTO)}')

# Funciones de lookup con cascada 3 niveles
def obtener_tarifa(prov, concepto):
    key = (normalizar(prov), concepto)
    return TARIFAS_PROV_CONCEPTO.get(key) or TARIFAS_CONCEPTO.get(concepto) or TARIFA_GLOBAL

def obtener_lag1(prov, concepto):
    key = (normalizar(prov), concepto)
    return LAG1_PROV_CONCEPTO.get(key) or LAG1_CONCEPTO.get(concepto) or MEDIANA_GLOBAL_LOG

def obtener_lag2(prov, concepto):
    key = (normalizar(prov), concepto)
    return LAG2_PROV_CONCEPTO.get(key) or LAG1_CONCEPTO.get(concepto) or MEDIANA_GLOBAL_LOG

def obtener_lag3(prov, concepto):
    key = (normalizar(prov), concepto)
    return LAG3_PROV_CONCEPTO.get(key) or LAG1_CONCEPTO.get(concepto) or MEDIANA_GLOBAL_LOG

def obtener_rolling3(prov, concepto):
    key = (normalizar(prov), concepto)
    return ROLLING3_PROV_CONCEPTO.get(key) or LAG1_CONCEPTO.get(concepto) or MEDIANA_GLOBAL_LOG

def obtener_n_obs(prov, concepto):
    return N_OBS_PROV_CONCEPTO.get((normalizar(prov), concepto), 0)

In [ ]:
# CELDA 3 — construir_features (v3: incluye Bloque B completo — lag2/lag3/rolling3/CV/premium_idx)
# Sincronizado con demo_prediccion.py v3

INCOTERM_GRUPO = {'E': 'GRUPO_E', 'F': 'GRUPO_F', 'C': 'GRUPO_C', 'D': 'GRUPO_D'}

RUTA_ORIGEN = {
    'VALENCIA': 'EUROPA', 'BARCELONA': 'EUROPA', 'ROTTERDAM': 'EUROPA',
    'HAMBURG': 'EUROPA', 'VIGO': 'EUROPA', 'RIGA': 'EUROPA', 'TALLIN': 'EUROPA',
    'LEIXOES': 'EUROPA', 'LISBON': 'EUROPA', 'BILBAO': 'EUROPA', 'GENOVA': 'EUROPA',
    'VALPARAISO': 'LATAM', 'SAN ANTONIO': 'LATAM', 'BUENOS AIRES': 'LATAM',
    'MANZANILLO': 'LATAM', 'GUAYAQUIL': 'LATAM', 'SANTIAGO': 'LATAM',
    'MIAMI': 'NORTEAM', 'LOS ANGELES': 'NORTEAM', 'NEW YORK': 'NORTEAM',
    'HOUSTON': 'NORTEAM', 'SEATTLE': 'NORTEAM',
    'SHANGHAI': 'ASIA', 'HONG KONG': 'ASIA', 'SINGAPORE': 'ASIA',
    'BUSAN': 'ASIA', 'GUANGZHOU': 'ASIA',
}
DIAS_TRANSITO_MAP = {'AIR': 5, 'LCL': 22, 'FCL': 28, 'COURIER': 3}


def mapear_incoterm(s):
    primera = normalizar(str(s))[:1]
    return INCOTERM_GRUPO.get(primera, 'GRUPO_F')


def mapear_ruta(pol):
    pol_up = normalizar(str(pol))
    for clave, region in RUTA_ORIGEN.items():
        if clave in pol_up:
            return region
    return 'OTROS'


def estimar_dias(mode):
    m = normalizar(str(mode))
    for k, v in DIAS_TRANSITO_MAP.items():
        if k in m:
            return v
    return 25


def _aplicar_categorias(X):
    """Fuerza dtype category con los niveles exactos del training en cada columna."""
    for col in CATEGORICAS:
        if col not in X.columns:
            continue
        niveles = CAT_LEVELS.get(col, ['DESCONOCIDO'])
        valores = X[col].fillna('DESCONOCIDO').astype(str)
        valores = valores.apply(lambda v: v if v in niveles else 'DESCONOCIDO')
        X[col] = pd.Categorical(valores, categories=niveles)
    return X


def construir_features(despacho):
    """
    Construye un DataFrame con una fila por concepto canónico para un despacho nuevo.
    Incluye todas las features del Bloque B: lag2/lag3, rolling3, provider_premium_idx,
    n_obs_prov_concepto, concepto_cv, te_proveedor, te_concepto_incoterm, te_concepto_ruta.
    """
    eta = pd.to_datetime(despacho.get('fecha_eta', datetime.now()), errors='coerce')
    if pd.isna(eta):
        eta = pd.Timestamp.now()

    mes      = eta.month
    semana   = eta.isocalendar()[1]
    mode     = normalizar(despacho.get('mode', 'FCL'))
    incoterm = normalizar(despacho.get('incoterm', 'FOB'))
    pol      = normalizar(despacho.get('pol', 'DESCONOCIDO'))
    proveedor = normalizar(despacho.get('proveedor_servicio',
                           despacho.get('proveedor_principal', 'DESCONOCIDO')))
    agencia   = normalizar(despacho.get('agencia_aduana', 'DESCONOCIDO'))

    contenedores = float(despacho.get('contenedores', 1) or 1)
    bultos       = float(despacho.get('bultos', 0) or 0)
    peso_bruto   = float(despacho.get('peso_kg', 0) or 0)

    incoterm_grupo  = mapear_incoterm(incoterm)
    ruta_origen     = mapear_ruta(pol)
    dias_transito   = estimar_dias(mode)
    densidad_bultos = (bultos / contenedores) if contenedores > 0 else 0
    carga_peso      = (peso_bruto / contenedores) if contenedores > 0 else 0
    frecuencia_prov = FRECUENCIA_PROVEEDOR.get(proveedor, 0)

    rows = []
    for concepto in CONCEPTOS:
        tarifa   = obtener_tarifa(proveedor, concepto)
        lag1     = obtener_lag1(proveedor, concepto)
        lag2     = obtener_lag2(proveedor, concepto)
        lag3     = obtener_lag3(proveedor, concepto)
        rolling3 = obtener_rolling3(proveedor, concepto)
        n_obs    = obtener_n_obs(proveedor, concepto)

        te_ci_key = f'{concepto}_{incoterm_grupo}'
        te_cr_key = f'{concepto}_{ruta_origen}'
        te_ci  = TE_CONCEPTO_INCOTERM.get(te_ci_key, _TE_CI_GLOBAL)
        te_cr  = TE_CONCEPTO_RUTA.get(te_cr_key, _TE_CR_GLOBAL)
        te_prov = TE_PROVEEDOR.get(proveedor, _TE_PROV_GLOBAL)

        med_conc = MEDIANA_POR_CONCEPTO.get(concepto, MEDIANA_GLOBAL_LOG)
        med_cm   = MEDIANA_CONCEPTO_MODE.get((concepto, mode), med_conc)
        diff_tar = tarifa - med_conc
        prem_idx = lag1 - med_conc
        cv_conc  = CONCEPTO_CV.get(concepto, 0.0)

        row = {
            # Temporal
            'año':               eta.year,
            'mes':               mes,
            'trimestre':         (mes - 1) // 3 + 1,
            'semana_año':        semana,
            'dia_semana':        eta.dayofweek,
            'dias_desde_inicio': max((eta - FECHA_INICIO).days, 0),
            'mes_sin':           np.sin(2 * np.pi * mes / 12),
            'mes_cos':           np.cos(2 * np.pi * mes / 12),
            'semana_sin':        np.sin(2 * np.pi * semana / 52),
            'semana_cos':        np.cos(2 * np.pi * semana / 52),
            'es_temporada_alta': int(8 <= mes <= 12),
            'es_cierre_fiscal':  int(mes in [11, 12]),
            # Logística
            'dias_transito':     dias_transito,
            'densidad_bultos':   densidad_bultos,
            'carga_peso':        carga_peso,
            'tiene_proyecto':    int(bool(despacho.get('proyecto'))),
            # Históricas — resueltas desde lookups reales del train
            'tarifa_historica':    tarifa,
            'lag1_costo':          lag1,
            'lag2_costo':          lag2,
            'lag3_costo':          lag3,
            'rolling_median_3':    rolling3,
            'frecuencia_proveedor': frecuencia_prov,
            # Target encoding (maps fit en train only — sin leakage)
            'te_concepto_incoterm': te_ci,
            'te_concepto_ruta':     te_cr,
            'te_proveedor':         te_prov,
            # Bloque B: interacciones y diferenciación por proveedor
            'median_concepto_mode': med_cm,
            'diff_tarifa_mediana':  diff_tar,
            'provider_premium_idx': prem_idx,
            'n_obs_prov_concepto':  n_obs,
            'concepto_cv':          cv_conc,
            # Categóricas
            'concepto_canonico':   concepto,
            'incoterm_grupo':      incoterm_grupo,
            'ruta_origen':         ruta_origen,
            'proveedor_norm':      proveedor,
            'agencia_aduana_norm': agencia,
            'mode':                mode,
            'type':                normalizar(despacho.get('type', 'FCL')),
        }
        rows.append(row)

    X = pd.DataFrame(rows)

    # Imputar numéricas faltantes con medianas del training
    for col in NUMERICAS:
        if col not in X.columns:
            X[col] = regresion_medians.get(col, 0.0)
        else:
            X[col] = X[col].fillna(regresion_medians.get(col, 0.0))

    # Aplicar dtype category con los niveles exactos del training
    X = _aplicar_categorias(X)

    return X

In [ ]:
# CELDA 4 — predecir_despacho (v3: bias 2 capas + CQR con estructura margen_inf/margen_sup)
# Corrige:
#   - _aplicar_bias: maneja (edges, ratios) como lista, no dict; aplica capa 2 por proveedor
#   - CQR: usa cqr_margenes[concepto]['margen_inf'] / ['margen_sup'] (no un float simple)

def aplicar_bias(pred_pen, concepto, proveedor=None):
    """Capa 1: piecewise por concepto. Capa 2: ratio por proveedor×concepto."""
    corr = bias_piecewise_corr.get(concepto, bias_escalar.get(concepto, 1.0))
    if isinstance(corr, tuple):
        edges, ratios = corr
        bucket = int(np.clip(np.searchsorted(edges[1:-1], pred_pen), 0, len(ratios) - 1))
        pred_l1 = pred_pen * ratios[bucket]
    else:
        pred_l1 = pred_pen * corr
    if proveedor is not None:
        ratio_l2 = bias_prov_concepto.get((str(proveedor).upper(), concepto), 1.0)
        return round(pred_l1 * ratio_l2, 2)
    return round(pred_l1, 2)


def calcular_riesgo_score(X_row):
    try:
        cat_cols = [c for c in CATEGORICAS if c in X_row.columns]
        num_cols = [c for c in NUMERICAS  if c in X_row.columns]
        X_num = X_row[num_cols].fillna(0).values
        X_cat = riesgo_enc.transform(X_row[cat_cols].astype(str))
        X_if  = np.hstack([X_num, X_cat])
        raw   = -float(iso_model.decision_function(X_if).ravel()[0])
        s_min = score_params['min']
        s_max = score_params['max']
        score_norm = float(np.clip((raw - s_min) / (s_max - s_min + 1e-9) * 100, 0, 100))
        nivel = 'ALTO' if score_norm >= 80 else ('MEDIO' if score_norm >= 50 else 'BAJO')
        return round(score_norm, 1), nivel
    except Exception:
        return 50.0, 'MEDIO'


def predecir_despacho(despacho):
    """Pipeline completo: construye features → predice costo + intervalos CQR + riesgo."""
    X = construir_features(despacho)
    resultados = []

    for concepto in CONCEPTOS:
        if concepto not in modelos_regresion:
            continue
        X_row  = X[X['concepto_canonico'] == concepto]
        if len(X_row) == 0:
            continue
        X_feat = X_row[[c for c in FEATURES if c in X_row.columns]]

        # Prediccion central con bias en 2 capas
        pred_log  = float(modelos_regresion[concepto].predict(X_feat)[0])
        pred_pen  = float(np.expm1(pred_log))
        proveedor = str(X_row['proveedor_norm'].iloc[0]) if 'proveedor_norm' in X_row.columns else None
        costo     = aplicar_bias(pred_pen, concepto, proveedor)

        # Intervalos CQR: margen relativo desde dict margen_inf/margen_sup
        if concepto in modelos_p10:
            p10_raw = float(np.expm1(modelos_p10[concepto].predict(X_feat)[0]))
            p90_raw = float(np.expm1(modelos_p90[concepto].predict(X_feat)[0]))
            marg = cqr_margenes.get(concepto, {'margen_inf': 0.20, 'margen_sup': 0.20})
            p10  = round(max(p10_raw * (1 - marg['margen_inf']), 0), 2)
            p90  = round(p90_raw * (1 + marg['margen_sup']), 2)
        else:
            p10 = round(costo * 0.70, 2)
            p90 = round(costo * 1.50, 2)

        riesgo_score, nivel_riesgo = calcular_riesgo_score(X_row)

        resultados.append({
            'concepto':               concepto,
            'costo_predicho_pen':     costo,
            'intervalo_inferior_pen': p10,
            'intervalo_superior_pen': p90,
            'riesgo_score_0_100':     riesgo_score,
            'nivel_riesgo':           nivel_riesgo,
        })

    df_res    = pd.DataFrame(resultados)
    total     = df_res['costo_predicho_pen'].sum()
    total_p10 = df_res['intervalo_inferior_pen'].sum()
    total_p90 = df_res['intervalo_superior_pen'].sum()
    return df_res, total, total_p10, total_p90

In [ ]:
# CELDA 5 — Funcion de presentacion del resultado
def mostrar_resultado(despacho, df_res, total, p10, p90):
    print('=' * 70)
    print(f'REPORTE DE PROVISION — {despacho.get("id_despacho", "NUEVO")}')
    print(f'Fecha ETA: {despacho.get("fecha_eta", "N/A")} | '
          f'Proveedor: {despacho.get("proveedor_principal", "N/A")}')
    print(f'Ruta: {despacho.get("pol", "?")} -> {despacho.get("pod", "?")} | '
          f'Modal: {despacho.get("modalidad", "?")} | '
          f'Incoterm: {despacho.get("incoterm_familia", despacho.get("incoterm", "?"))}')
    print('=' * 70)

    df_show = df_res.sort_values('costo_predicho_pen', ascending=False)
    print(f'\n{"Concepto":<40} {"Costo":>10} {"P10":>10} {"P90":>10} {"Riesgo":>8}')
    print('-' * 82)
    for _, row in df_show.iterrows():
        print(f'{row["concepto"]:<40} '
              f'S/{row["costo_predicho_pen"]:>8,.0f} '
              f'S/{row["intervalo_inferior_pen"]:>8,.0f} '
              f'S/{row["intervalo_superior_pen"]:>8,.0f} '
              f'{row["nivel_riesgo"]:>8}')

    print('-' * 82)
    print(f'{"TOTAL":<40} S/{total:>8,.0f} S/{p10:>8,.0f} S/{p90:>8,.0f}')

    alto_riesgo = df_res[df_res['nivel_riesgo'] == 'ALTO']
    if len(alto_riesgo) > 0:
        print(f'\nALERTA: {len(alto_riesgo)} conceptos con riesgo ALTO — revisar:')
        for _, r in alto_riesgo.iterrows():
            print(f'  - {r["concepto"]}: score={r["riesgo_score_0_100"]:.0f}/100')

    # Indicador de calidad del intervalo
    ancho_rel = (p90 - p10) / total if total > 0 else 0
    if ancho_rel > 1.5:
        calidad = 'ANCHO  — alta incertidumbre, usar con cautela'
    elif ancho_rel > 0.8:
        calidad = 'MODERADO — rango aceptable para provision'
    else:
        calidad = 'PRECISO — confianza alta en la estimacion'

    print(f'\nINTERVALO 90% CONFIANZA: S/{p10:,.0f} — S/{p90:,.0f}')
    print(f'Ancho del intervalo: {ancho_rel*100:.1f}% del total  [{calidad}]')

    if bias_correction:
        bias_vals = [bias_correction.get(c, 1.0) for c in df_res['concepto']]
        bias_med  = round(float(np.median(bias_vals)), 2)
        print(f'Corrección de sesgo aplicada (bias mediano: {bias_med:.2f}x)')

    print('=' * 70)


In [ ]:
# CELDA 6 — Ejemplo 1: Despacho maritimo europeo
despacho_1 = {
    'id_despacho':         '26-099-HPER',
    'proveedor_servicio':  'AIRSEALOG',
    'proveedor_principal': 'PROJAR',
    'agencia_aduana':      'AVM ADUANERA',
    'acreedor':            'AVM ADUANERA',
    'pol':                 'VALENCIA',
    'pod':                 'CALLAO',
    'modalidad':           'SEA / FCL',
    'incoterm_familia':    'GRUPO_C',
    'contenedores':        2,
    'bultos':              480,
    'peso_kg':             24000,
    'fecha_eta':           '2026-09-15',
    'proyecto':            None,
}

df_res_1, total_1, p10_1, p90_1 = predecir_despacho(despacho_1)
mostrar_resultado(despacho_1, df_res_1, total_1, p10_1, p90_1)


In [ ]:
# CELDA 7 — Ejemplo 2: Despacho aereo express
despacho_2 = {
    'id_despacho':         'DEMO-AIR-001',
    'proveedor_servicio':  'AIRSEALOG',
    'proveedor_principal': 'DUNAVANT',
    'agencia_aduana':      'AVM ADUANERA',
    'pol':                 'SANTIAGO',
    'pod':                 'CALLAO',
    'modalidad':           'AIR / AIR',
    'incoterm_familia':    'GRUPO_E',
    'contenedores':        0,
    'bultos':              12,
    'peso_kg':             480,
    'fecha_eta':           '2026-10-03',
    'proyecto':            'PROJ-2026-007',
}

df_res_2, total_2, p10_2, p90_2 = predecir_despacho(despacho_2)
mostrar_resultado(despacho_2, df_res_2, total_2, p10_2, p90_2)


In [ ]:
# CELDA 8 — Validacion COMPLETA en todo el test set
# CAMBIO L: reemplaza los 3 operaciones aleatorias (inestable) por evaluacion sobre
# todas las operaciones del test no superpuestas (~200 ops). Produce MAPE estadisticamente confiable.

test = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')
test = test[test['Importe_Total_PEN'] > 0].copy()

train_all   = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
ops_overlap = set(train_all['Nro. Ope.'].unique()) & set(test['Nro. Ope.'].unique())
test_eval_v = test[~test['Nro. Ope.'].isin(ops_overlap)].copy()
ops_all     = test_eval_v['Nro. Ope.'].unique()

print(f'Evaluando {len(ops_all)} operaciones del test set (sin solapamiento con train)...\n')

errores_pct  = []
ratios_bias  = []
dentro_count = 0
resultados_v = []

for op_id in ops_all:
    sub     = test_eval_v[test_eval_v['Nro. Ope.'] == op_id]
    primera = sub.iloc[0]
    total_real = sub['Importe_Total_PEN'].sum()

    if total_real < 100:  # excluir operaciones triviales
        continue

    despacho_real = {
        'id_despacho':         op_id,
        'proveedor_servicio':  primera.get('Proveedor_norm', 'DESCONOCIDO'),
        'proveedor_principal': primera.get('Proveedor Principal_norm', 'DESCONOCIDO'),
        'agencia_aduana':      primera.get('AGENCIA DE ADUANA_norm', 'DESCONOCIDO'),
        'acreedor':            primera.get('ACREEDOR_norm', 'DESCONOCIDO'),
        'pol':                 primera.get('POL', ''),
        'pod':                 primera.get('POD', ''),
        'modalidad':           primera.get('Modalidad (MODE Y TYPE)', ''),
        'incoterm_familia':    primera.get('incoterm_familia', ''),
        'contenedores':        primera.get('Cantidad de Contenedores', 0),
        'bultos':              primera.get('Cantidad de Bultos (BULKS)', 0),
        'peso_kg':             primera.get('Peso Bruto (kg)', 0),
        'fecha_eta':           str(primera.get('Fecha_Imputada', datetime.now()))[:10],
    }

    try:
        df_pred, total_pred, p10_pred, p90_pred = predecir_despacho(despacho_real)
    except Exception:
        continue

    error_pct = abs(total_pred - total_real) / total_real * 100
    dentro    = (p10_pred <= total_real <= p90_pred)
    ratio     = total_real / total_pred if total_pred > 0 else 1.0

    errores_pct.append(error_pct)
    ratios_bias.append(ratio)
    if dentro:
        dentro_count += 1

    resultados_v.append({
        'op': op_id, 'real': total_real, 'pred': total_pred,
        'error_pct': error_pct, 'dentro': dentro, 'ratio': ratio,
    })

df_eval = pd.DataFrame(resultados_v)
n_eval  = len(df_eval)

# --- Resumen estadístico ---
mape_med = np.median(errores_pct)
mape_p75 = np.percentile(errores_pct, 75)
cob_pct  = dentro_count / n_eval * 100
bias_med = np.median(ratios_bias)
sesgo    = 'subestimacion' if bias_med > 1.05 else ('sobreestimacion' if bias_med < 0.95 else 'sin sesgo')

print(f'VALIDACION COMPLETA — {n_eval} operaciones (ops triviales excluidas)')
print(f'{'='*55}')
print(f'MAPE mediano:          {mape_med:.1f}%  (antes: 31.1%)')
print(f'MAPE percentil 75:     {mape_p75:.1f}%')
print(f'Cobertura 90%:         {dentro_count}/{n_eval} ({cob_pct:.1f}%)')
print(f'Factor Real/Pred:      {bias_med:.3f}x  [{sesgo}]  (antes: 1.45x)')
print(f'Bias correction activo: {"SI (piecewise)" if bias_correction_piecewise else "SI (escalar)" if bias_correction else "NO"}')
print()

# --- Mostrar las 5 operaciones con mayor error ---
print('Top 5 operaciones con mayor error:')
top5 = df_eval.nlargest(5, 'error_pct')[['op', 'real', 'pred', 'error_pct', 'ratio']]
for _, r in top5.iterrows():
    print(f'  {r["op"]}: Real=S/{r["real"]:,.0f} Pred=S/{r["pred"]:,.0f} '
          f'Error={r["error_pct"]:.1f}% Ratio={r["ratio"]:.2f}x')

# --- Histograma de errores ---
plt.figure(figsize=(10, 4))
plt.hist(errores_pct, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
plt.axvline(mape_med, color='red', linestyle='--', linewidth=2,
            label=f'Mediana={mape_med:.1f}%')
plt.axvline(mape_p75, color='orange', linestyle='--', linewidth=1.5,
            label=f'P75={mape_p75:.1f}%')
plt.xlabel('Error % por operación')
plt.ylabel('Frecuencia')
plt.title(f'Distribución de errores — test set completo ({n_eval} operaciones)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CELDA 9 — Widget interactivo
import ipywidgets as widgets
from IPython.display import display, clear_output

w_id   = widgets.Text(value='DEMO-001', description='ID:')
w_prov = widgets.Text(value='AIRSEALOG', description='Proveedor:')
w_ppal = widgets.Text(value='PROJAR', description='Ppal:')
w_ag   = widgets.Text(value='AVM ADUANERA', description='Agencia:')
w_pol  = widgets.Dropdown(options=['VALENCIA','SANTIAGO','COLOMBO','SAN ANTONIO','MIAMI','OTRO'],
                          value='VALENCIA', description='POL:')
w_pod  = widgets.Text(value='CALLAO', description='POD:')
w_mod  = widgets.Dropdown(options=['SEA / FCL','SEA / LCL','AIR / AIR','COURIER'],
                          value='SEA / FCL', description='Modalidad:')
w_inc  = widgets.Dropdown(options=['GRUPO_C','GRUPO_E','GRUPO_F','GRUPO_D'],
                          value='GRUPO_C', description='Incoterm:')
w_cont = widgets.IntText(value=2, description='Contenedores:')
w_bult = widgets.IntText(value=480, description='Bultos:')
w_peso = widgets.FloatText(value=24000, description='Peso (kg):')
w_eta  = widgets.Text(value=datetime.now().strftime('%Y-%m-%d'), description='Fecha ETA:')
btn    = widgets.Button(description='Predecir', button_style='primary')
out    = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        d = {
            'id_despacho': w_id.value, 'proveedor_servicio': w_prov.value,
            'proveedor_principal': w_ppal.value, 'agencia_aduana': w_ag.value,
            'pol': w_pol.value, 'pod': w_pod.value,
            'modalidad': w_mod.value, 'incoterm_familia': w_inc.value,
            'contenedores': w_cont.value, 'bultos': w_bult.value,
            'peso_kg': w_peso.value, 'fecha_eta': w_eta.value,
        }
        try:
            df_r, tot, p10, p90 = predecir_despacho(d)
            mostrar_resultado(d, df_r, tot, p10, p90)
        except Exception as e:
            print(f'Error: {e}')
            import traceback; traceback.print_exc()

btn.on_click(on_click)
grid = widgets.GridBox(
    [w_id, w_prov, w_ppal, w_ag, w_pol, w_pod, w_mod, w_inc, w_cont, w_bult, w_peso, w_eta],
    layout=widgets.Layout(grid_template_columns='repeat(3, 280px)')
)
display(widgets.VBox([grid, btn, out]))
